In [1]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [2]:
data = np.load("pems04.npz")

traffic = data["data"][:,:,2]

print(traffic.shape)

(16992, 307)


In [3]:
scaler = MinMaxScaler()

traffic_scaled = scaler.fit_transform(
    traffic
)

print(
    traffic_scaled.shape
)

(16992, 307)


In [4]:
X = []
y = []

sequence_length = 12

for i in range(
    len(traffic_scaled)
    -
    sequence_length
):

    X.append(
        traffic_scaled[
            i:i+sequence_length
        ]
    )

    y.append(
        traffic_scaled[
            i+sequence_length
        ]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(16980, 12, 307)
(16980, 307)


In [5]:
split = int(
    len(X) * 0.8
)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

print(X_train.shape)
print(X_test.shape)

(13584, 12, 307)
(3396, 12, 307)


In [6]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [7]:
train_dataset = TensorDataset(
    X_train,
    y_train
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [8]:
import torch
import torch.nn as nn

class MultiStepTransformer(nn.Module):

    def __init__(
        self,
        num_nodes=307,
        d_model=128,
        nhead=4,
        num_layers=2
    ):

        super().__init__()

        self.input_proj = nn.Linear(
            num_nodes,
            d_model
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.fc = nn.Linear(
            d_model,
            num_nodes
        )

    def forward(self,x):

        x = self.input_proj(x)

        x = self.transformer(x)

        x = x.mean(dim=1)

        x = self.fc(x)

        return x

In [9]:
model = MultiStepTransformer()

X_batch, y_batch = next(iter(train_loader))

pred = model(X_batch)

print(pred.shape)
print(y_batch.shape)

torch.Size([64, 307])
torch.Size([64, 307])


In [10]:
model = MultiStepTransformer()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

import time

train_start = time.time()

epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(
            pred,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs}",
        f"Loss: {total_loss/len(train_loader):.6f}"
    )

Epoch 1/30 Loss: 0.034271
Epoch 2/30 Loss: 0.008974
Epoch 3/30 Loss: 0.007360
Epoch 4/30 Loss: 0.006397
Epoch 5/30 Loss: 0.005668
Epoch 6/30 Loss: 0.005236
Epoch 7/30 Loss: 0.004934
Epoch 8/30 Loss: 0.004690
Epoch 9/30 Loss: 0.004426
Epoch 10/30 Loss: 0.004240
Epoch 11/30 Loss: 0.004033
Epoch 12/30 Loss: 0.003917
Epoch 13/30 Loss: 0.003754
Epoch 14/30 Loss: 0.003614
Epoch 15/30 Loss: 0.003519
Epoch 16/30 Loss: 0.003383
Epoch 17/30 Loss: 0.003292
Epoch 18/30 Loss: 0.003178
Epoch 19/30 Loss: 0.003113
Epoch 20/30 Loss: 0.003025
Epoch 21/30 Loss: 0.002951
Epoch 22/30 Loss: 0.002897
Epoch 23/30 Loss: 0.002809
Epoch 24/30 Loss: 0.002722
Epoch 25/30 Loss: 0.002689
Epoch 26/30 Loss: 0.002630
Epoch 27/30 Loss: 0.002569
Epoch 28/30 Loss: 0.002502
Epoch 29/30 Loss: 0.002496
Epoch 30/30 Loss: 0.002433


In [11]:
train_time = time.time() - train_start

print("Time Taken:", train_time)

Time Taken: 697.0043702125549


In [12]:
torch.save(
    model.state_dict(),
    "MultiStepTransformer-PEMS04.pth"
)

In [13]:
model.eval()

infer_start = time.time()

with torch.no_grad():

    predictions = model(
        X_test
    )

predictions = predictions.numpy()

infer_time = time.time() - infer_start
print("Infer Time:", infer_time)

true_values = y_test.numpy()

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

Infer Time: 1.0639193058013916
MAE: 0.040618744
RMSE: 0.07039359


In [14]:
from sklearn.metrics import r2_score

mape = np.mean(
    np.abs((true_values - predictions) /
           np.maximum(np.abs(true_values), 1e-6))
) * 100

r2 = r2_score(
    true_values.flatten(),
    predictions.flatten()
)
print("MAPE:", mape)
print("R2:", r2)

MAPE: 6570.8160400390625
R2: 0.764364351945461


In [15]:
params = sum(
    p.numel()
    for p in model.parameters()
)

print("Parameters:", params)

Parameters: 1265075
